In [ ]:
# Firstly We load the dataset

import pandas as pd
df = pd.read_csv("cars.csv")

# Then we explore the dataset
# how big is it 
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

# numbers or texts
print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
print(df.head())


Dataset shape: (653721, 56)

Columns:
['id_x', 'car_rel_url_x', 'datetime_scrape', 'name', 'price_x', 'currency_x', 'datetime_product', 'city', 'day', 'hour', 'attributes', 'production_year', 'engine_displacement_num', 'engine_displacement_unit', 'kilometrage_num', 'kilometrage_unit', 'barter', 'loan', 'salon', 'spare_parts', 'vip', 'featured', 'img_url', 'id_y', 'cars_id', 'car_rel_url_y', 'datetime', 'description', 'price_y', 'currency_y', 'owner_name', 'shop_name', 'phone', 'updated', 'views', 'vin', 'car_details_id_x', 'Ban növü', 'Buraxılış ili', 'Hansı bazar üçün yığılıb', 'Marka', 'Model', 'Mühərrik', 'Qəzalı', 'Rəng', 'Sahiblər', 'Sürətlər qutusu', 'Vəziyyəti', 'Yeni', 'Yerlərin sayı', 'Yürüş', 'Ötürücü', 'Şəhər', 'car_details_id_y', 'car_rel_url', 'extra_info']

Data types:
id_x                         object
car_rel_url_x                object
datetime_scrape              object
name                         object
price_x                     float64
currency_x                

## Checkpoint 1: Descriptive Statistics for Groups

The task require calculating descriptive statistics (mean, median, standard deviation, and distribution shape) for different groups in dataset.

The dataset contains several categorical variables that can be used to create groups. For this analysis, Marka (car brand) was selected to be the grouping variable. Car brands are useful groups because they allow us to compare how car prices differ between manufacturers.

The variable price_x was selected for the analysis because it contains the car prices, which is the numerical value we want to compare between these groups.

The following descriptive statistics were calculated for each car brand:
Mean: average price of cars in the group.
Median: middle price value in the group.
Standard deviation: shows the spread of prices within the group.
Skewness: shows the shape of the price distribution.


In [2]:
group_stats = df.groupby("Marka")["price_x"].agg(
    Mean="mean",
    Median="median",
    Standard_Deviation="std",
    Distribution_Shape="skew"
)

group_stats

,Mean,Median,Standard_Deviation,Distribution_Shape
Marka,,,,
ATV,3508.333333,3950.0,1691.276638,-0.814495
Abarth,16694.495413,18100.0,3071.518953,-0.960685
Acura,18007.575758,18000.0,2837.145386,-1.798210
Alfa Romeo,65978.372591,54600.0,31553.669504,0.403725
Aprilia,17914.191919,8399.0,9774.995984,0.069288
...,...,...,...,...
ZXMCO,2625.000000,3500.0,1371.039751,-1.064886
Zamyad,38700.000000,38700.0,NaN,NaN
Zongshen,4733.166667,5799.5,2355.325915,-0.711247


## Checkpoint 2: Hypothesis Formulation

Business Question 1:
Do luxury car brand has a higher average price than non-luxury car brand?

H0 (Null Hypothesis): 
There is no significant difference in the average price between luxury and non-luxury car brand.

H1 (Alternative Hypothesis): 
Luxury car brands have a significantly higher average price than non-luxury car brands.


Business Question 2:
Does the average car price differ across different production years?

H0 (Null Hypothesis):
The mean car price is the same across all production years.

H1 (Alternative Hypothesis):
At least one production year has a different mean car price.


## Data Preparation

To answer the first business question, car listings are divided into two groups:
luxury and non-luxury brands.

To answer the second business question, car listings are grouped by production year.

These groups will be analyzed in the next step using appropriate statistical tests.

In [ ]:
# Checking car brand

print("Number of unique brands:", df["Marka"].nunique())

print("\nTop brands:")
print(df["Marka"].value_counts().head(15))


# next check production years 

print("\nProduction year values:")
print(df["production_year"].value_counts().sort_index())


Number of unique brands: 208

Top brands:
Marka
Mercedes      90830
Hyundai       85968
Toyota        75167
Kia           67029
BMW           54432
Land Rover    29883
Ford          27159
Nissan        25043
LADA (VAZ)    24014
Chevrolet     19163
Opel          15023
Lexus         11805
Changan       11164
Volkswagen     9348
Mitsubishi     8292
Name: count, dtype: int64

Production year values:
production_year
0        5788
1938        2
1947        1
1954       63
1955        2
        ...  
2020    16721
2021    18133
2022    17904
2023    27001
2024    70223
Name: count, Length: 73, dtype: int64


In [ ]:
# Luxury vs non-luxury groups
# Luxury brands were selected based on commonly recognized premium car manufacturers.
# Prices are extracted from each group because the test compares average prices.

luxury_brands = [
    "Mercedes", "BMW", "Audi", "Lexus",
    "Porsche", "Land Rover", "Jaguar",
    "Bentley", "Maserati", "Ferrari"
]

luxury = df[df["Marka"].isin(luxury_brands)]["price_x"].dropna()

non_luxury = df[~df["Marka"].isin(luxury_brands)]["price_x"].dropna()

print("Luxury cars:", len(luxury))
print("Non-luxury cars:", len(non_luxury))


# Production year groups
# Inspection showed that production_year contains 0 values
# These records are removed because 0 is not a valid vehicle production year

df_clean = df[df["production_year"] > 1900]

year_groups = [
    group["price_x"].dropna()
    for _, group in df_clean.groupby("production_year")
]

print("Number of production year groups:", len(year_groups))

## Checkpoint 3: Selection and Execution of Statistical Tests

Business Question 1:
Do luxury car brands have a higher average price than non-luxury car brands?

To answer this question, the variables were checked:

Target variable: price_x
Type: Numerical (continuous)
We want to compare average prices.

Grouping variable: car category (luxury vs non-luxury)
Type: Categorical
Contains two independent groups.

Since the goal is to compare the mean of a numerical variable between two independent groups, an independent two-sample t-test was selected.

Business Question 2:
Does the average car price differ across different production years?

Variables:

Target variable: price_x
Type: Numerical (continuous)

Grouping variable: production_year
Type: Categorical after grouping by year
Contains more than two independent groups.

Since the goal is to compare the mean price across multiple groups, a one-way ANOVA test was selected.


In [4]:
# Execution of statistical tests 
from scipy.stats import ttest_ind, f_oneway

# Luxury vs non-luxury groups

luxury_brands = [
    "Mercedes", "BMW", "Audi", "Lexus",
    "Porsche", "Land Rover", "Jaguar",
    "Bentley", "Maserati", "Ferrari"
]

luxury = df[df["Marka"].isin(luxury_brands)]["price_x"].dropna()

non_luxury = df[~df["Marka"].isin(luxury_brands)]["price_x"].dropna()


t_stat, p_value = ttest_ind(
    luxury,
    non_luxury,
    equal_var=False
)

print("T-test result:")
print("t-statistic:", t_stat)
print("p-value:", p_value)



T-test result:
t-statistic: 142.3057823838934
p-value: 0.0


In [5]:
df_clean = df[df["production_year"] > 1900]

year_groups = [
    group["price_x"].dropna()
    for _, group in df_clean.groupby("production_year")
]


anova_stat, anova_p = f_oneway(*year_groups)

print("ANOVA result:")
print("F-statistic:", anova_stat)
print("p-value:", anova_p)

ANOVA result:
F-statistic: 3560.843565666654
p-value: 0.0


## Checkpoint 4: Interpretation of p-value and Confidence Interval

The t-test and ANOVA tests both returned p-values close to 0. Since these values are smaller than the significance level of 0.05, we reject the null hypotheses.

1) For the first business question, this means there is a statistically significant difference between the average prices of luxury and non-luxury cars.

2) For the second business question, this means that average car prices are not the same across all production years, and at least one production year has a different average price.

To better understand the first result, a 95% confidence interval was calculated for the average prices of luxury and non-luxury cars. The confidence interval shows the range where the true average price is expected to be.

In [ ]:
from scipy import stats
import numpy as np

# 95% Confidence Interval for luxury cars
ci_luxury = stats.t.interval(
    confidence=0.95,
    df=len(luxury)-1,
    loc=np.mean(luxury),
    scale=stats.sem(luxury)
)

# 95% Confidence Interval for non-luxury cars
ci_non_luxury = stats.t.interval(
    confidence=0.95,
    df=len(non_luxury)-1,
    loc=np.mean(non_luxury),
    scale=stats.sem(non_luxury)
)

print("Luxury cars 95% CI:", ci_luxury)
print("Non-luxury cars 95% CI:", ci_non_luxury)

mean_difference = luxury.mean() - non_luxury.mean()

print("Mean price difference:", round(mean_difference, 2), "AZN")

Luxury cars 95% CI: (np.float64(40836.64548873269), np.float64(41190.73650082532))
Non-luxury cars 95% CI: (np.float64(27414.74175030478), np.float64(27531.93885623881))

Mean price difference (luxury - non-luxury): 13540.35 AZN

T-test p-value: 0.0
Mann-Whitney U p-value: 0.0
ANOVA p-value: 0.0
Kruskal-Wallis p-value: 0.0


## Checkpoint 4: Interpretation of p-value and Confidence Interval
The t-test and ANOVA both produced p-values smaller than 0.05, so the null hypotheses are rejected.

The calculated 95% confidence intervals show that the average price of luxury cars is approximately 40,837–41,191 AZN, while the average price of non-luxury cars is approximately 27,415–27,532 AZN.

The mean price difference is approximately 13,540 AZN, indicating that luxury cars have a higher average price than non-luxury cars.

## Checkpoint 5: Testing Assumptions Before Parametric Tests

Before using parametric statistical tests such as the independent t-test and one-way ANOVA, their assumptions should be checked.

The following assumptions were tested:
Normality: whether the price data follows a normal distribution
Equality of variances: whether the groups have similar variances

The Shapiro-Wilk test was used to test normality, and Levene's test was used to test equality of variances.

In [ ]:
from scipy.stats import shapiro, levene

# Normality test (sample of 5000 observations from each group)
shapiro_luxury = shapiro(luxury.sample(5000, random_state=42))
shapiro_non_luxury = shapiro(non_luxury.sample(5000, random_state=42))

print("Luxury cars - Shapiro-Wilk Test")
print(shapiro_luxury)

print("\nNon-luxury cars - Shapiro-Wilk Test")
print(shapiro_non_luxury)

# Equality of variances
levene_result = levene(luxury, non_luxury)

print("\nLevene's Test")
print(levene_result)

Shapiro-Wilk Test
Luxury cars: ShapiroResult(statistic=np.float64(0.6683507508110935), pvalue=np.float64(1.5940140517330788e-71))
Non-luxury cars: ShapiroResult(statistic=np.float64(0.7255909026469335), pvalue=np.float64(1.0316493954570618e-67))

Levene's Test (luxury vs non-luxury)
Statistic: 25426.24559970701
p-value: 0.0

Levene's Test (across production years)
Statistic: 1555.428181651275
p-value: 0.0


Interpretation:

The p-values from both Shapiro-Wilk tests are smaller than 0.05, indicating that the price data is not normally distributed.

The p-value from Levene's test is also smaller than 0.05, indicating that the variances of the two groups are not equal.

Therefore, the assumptions of the classical independent t-test are not fully satisfied.

## Checkpoint 6: Business Conclusion

The statistical tests showed that luxury cars have a significantly higher average price than non-luxury cars. They also showed that average car prices differ across production years. These results can help businesses make pricing and inventory decisions. For example, dealerships can consider brand category and production year when setting prices or evaluating the value of a vehicle.
The analysis shows that both brand category and production year are related to car prices, so they can be useful factors when making pricing decisions.